In [ ]:
#Import packages
import numpy as np
import pandas as pd

In [ ]:
#Loading the train data
train_data = pd.read_parquet('../input/amex-parquet/train_data.parquet')
                         
#Explore train data
train_data.head()

In [ ]:
#Check shape of train data
train_data.shape

In [ ]:
#Check for number of unique customers
len(train_data.customer_ID.unique())

## We have 458913 unique customers.

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
# Check for number of missing values
train_data.isnull().sum()

## Could be observed that there are many columns with many missing values

In [ ]:
# There are multiple transactions. Lets take only the latest transaction from each customer.
train=train_data.groupby('customer_ID').tail(1)
train=train.set_index(['customer_ID'])

#Drop date column since it is no longer relevant
train.drop(['S_2'],axis=1,inplace=True)
#Check for number of rows
train.shape
# We now have 458913 rows, which corresponds to the number of unique customers.

In [ ]:
#Identify columns which are not numeric
train.select_dtypes(['object'])

##D_63 and D_64 turns out to be categorical but are strings

In [ ]:
#Perform one-hot encoding for D_63 and D_64
#Drop columns D_63 and D_64 subsequently
train_D63 = pd.get_dummies(train[['D_63']])
train = pd.concat([train, train_D63], axis=1)
train = train.drop(['D_63'], axis=1)

train_D64 = pd.get_dummies(train[['D_64']])
train = pd.concat([train, train_D64], axis=1)
train = train.drop(['D_64'], axis=1)

In [ ]:
#Lets check for the columns
train.columns
# We now have 196 columns including target
# We need to reduce the dimensionality of the data

In [ ]:
#Given that there are many columns with large number of missing values, it is impractical to go through every single one of them to determine whether it is useful. 
#Furthermore, we do not have information on the feature (e.g. actual name of the feature) except the type of variable
#Lets remove columns if there are >85% of missing values
train=train.dropna(axis=1, thresh=int(0.85*len(train)))

#Checking the shape of new train data
train.shape
## We are left with 160 columns

In [ ]:
# We shall remove highly correlated features.
train_without_target=train.drop(['target'],axis=1)
cor_matrix = train_without_target.corr().abs()
upper_tri = cor_matrix.where((np.triu(np.ones(cor_matrix.shape), k=1) + np.tril(np.ones(cor_matrix.shape), k=-1)).astype(bool))
#Drop out columns with absolute correlation of more than 85%
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.85)]
train_drop_highcorr=train_without_target.drop(to_drop,axis=1)
train_drop_highcorr.shape
#We are now left with 131 columns (excluding target), which is still significant

In [ ]:
# Lets remove columns with variance less than or equal to 0.05. Keep only columns with high variance.
from sklearn.feature_selection import VarianceThreshold
from itertools import compress
def fs_variance(df, threshold:float=0.05):
    """
    Return a list of selected variables based on the threshold.
    """
    # The list of columns in the data frame
    features = list(df.columns)
    
    # Initialize and fit the method
    vt = VarianceThreshold(threshold = threshold)
    _ = vt.fit(df)
    
    # Get which column names which pass the threshold
    feat_select = list(compress(features, vt.get_support()))
    
    return feat_select
columns_to_keep=fs_variance(train_drop_highcorr)
# We are left with 85 columns (excluding target), which passed the threshold.
train_final=train[columns_to_keep]
len(columns_to_keep)

In [ ]:
#Concat target onto train_final
train_final1=train_final.join(train['target'])
x_train=train_final1.drop(['target'],axis=1)
y_train=train_final1['target']

In [ ]:
# Split train data into training and testing sets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score  
from sklearn.metrics import precision_score                         
from sklearn.metrics import recall_score
x_train_split, x_test_split, y_train_split, y_test_split = train_test_split(x_train, y_train, test_size=0.25, random_state=26)

In [ ]:
# Comment out this section as it takes a very long time to run, the results of randomizedsearchCV is included below
# Grid of hyperparameters to search over
#from sklearn.model_selection import RandomizedSearchCV
#param_random_gb = {'learning_rate': np.arange(0.05,0.55, 0.1),
#                   'n_estimators' : [125,150,175],
#                   'subsample' : np.arange(0.3,1.0, 0.1),
#                   'max_depth':[3,4,5]}

# Use XGBoost Classifier
from xgboost import XGBClassifier

# Perform RandomizedSearchCV
#mse_random = RandomizedSearchCV(estimator = XGBClassifier(), param_distributions = param_random_gb, 
#                               n_iter = 10,scoring = 'neg_mean_squared_error', cv = 4, verbose = 1)

#mse_random.fit(x_train_split,y_train_split)

#print("Best parameter: ", mse_random.best_params_)
#print("Lowest RMSE: ", np.sqrt(np.abs(mse_random.best_score_)))
#Best parameter:  {'subsample': 0.5, 'n_estimators': 175, 'max_depth': 3, 'learning_rate': 0.15}
#Lowest RMSE:  0.32263831733224874

In [ ]:
#Run XGBoost model with the best parameters found
model=XGBClassifier(n_estimators=200,max_depth=3,learning_rate=0.15, subsample=0.5)
model.fit(x_train_split,y_train_split)
#Test the model
y_predict=model.predict(x_test_split)
print('XGBoost Classifier Accuracy: {:.3f}'.format(accuracy_score(y_test_split, y_predict)))
# Achieved 89.5% accuracy

In [ ]:
print('\nXGBoost Classifier Precision: {:.3f}'.format(precision_score (y_test_split, y_predict)))
# Achieved Precision Score of 0.799

In [ ]:
print('\nXGBoost Classifier Recall: {:.3f}'.format(recall_score (y_test_split, y_predict)))
#Achieved Recall Score of 0.791

In [ ]:
# Make a list of columns that we want to load for test data. Remove one-hot encoded names and target (since these columns not in the test data)
columns_to_load=list(columns_to_keep)
columns_to_load=columns_to_load+['D_63','D_64','customer_ID','S_2']
columns_to_load.remove('D_63_CO')
columns_to_load.remove('D_63_CR')
columns_to_load.remove('D_63_CL')
columns_to_load.remove('D_64_O')
columns_to_load.remove('D_64_R')
columns_to_load.remove('D_64_U')

In [ ]:
#Read in the test_data
test_data = pd.read_parquet('../input/amex-parquet/test_data.parquet',columns=columns_to_load)

In [ ]:
# There are multiple transactions. Lets take only the latest transaction from each customer.
test=test_data.groupby('customer_ID').tail(1)
test=test.set_index(['customer_ID'])

#Drop date column since it is no longer relevant
test.drop(['S_2'],axis=1,inplace=True)

In [ ]:
#Perform one-hot encoding for D_63 and D_64
#Drop columns D_63 and D_64 subsequently
test_D63 = pd.get_dummies(test[['D_63']])
test = pd.concat([test, test_D63], axis=1)
test = test.drop(['D_63'], axis=1)

test_D64 = pd.get_dummies(test[['D_64']])
test = pd.concat([test, test_D64], axis=1)
test = test.drop(['D_64'], axis=1)

In [ ]:
#Keep columns that we want.
test_final=test[columns_to_keep]

In [ ]:
#Predict probabilities of default
y_test_predict=model.predict_proba(test_final)

In [ ]:
#Retrieve the probability of default
y_predict_final=y_test_predict[:,1]

# Merge the prediction and customer_ID into submission dataframe
submission = pd.DataFrame({"customer_ID":test_final.index,"prediction":y_predict_final})

submission.to_csv('submission.csv', index=False)